# Fine-tuning Gemma 3 Vision for PII Detection

This notebook follows the [official Google guide](https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora) to fine-tune a **Gemma 3 Vision** model for detecting PII in images using **QLoRA** and **SFTTrainer**.

### Pipeline Overview:
1. **Environment Setup**: Install latest transformers and multimodal libraries.
2. **Model Loading**: Load `google/gemma-3-4b-pt` with 4-bit quantization.
3. **Dataset Preparation**: Translate PaliGemma-style `metadata.jsonl` into simple text format.
4. **Training**: Use `SFTTrainer` with a custom vision-aware data collator.
5. **Inference**: Test the model on PII detection.

## 1. Setup and Installations

In [1]:
!pip install -q "torch>=2.4.0" "transformers>=4.51.3" "datasets==3.3.2" "accelerate==1.4.0" "bitsandbytes>=0.46.1" "trl==0.15.2" "peft==0.14.0" "jinja2>=3.1.0" pillow sentencepiece
import torch
import os
import json
import random
import glob
from PIL import Image
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

print("Setup complete.")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Setup complete.
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Authentication
Ensure you have access to `google/gemma-3-4b-pt` on Hugging Face.

In [2]:
from huggingface_hub import login

# Using the token found in the environment
login("hf_")

## 3. Load Model and Processor

In [3]:
model_id = "google/gemma-3-4b-pt"
processor_id = "google/gemma-3-4b-it"

print(f"Loading model: {model_id}...")
model_kwargs = dict(
    attn_implementation="eager", 
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_storage=torch.bfloat16,
)

model = AutoModelForImageTextToText.from_pretrained(model_id, trust_remote_code=True, **model_kwargs)
processor = AutoProcessor.from_pretrained(processor_id, trust_remote_code=True)

print("Model and Processor loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: google/gemma-3-4b-pt...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Model and Processor loaded.


## 4. Dataset Preparation
We use a plain text format for the **Base (PT) model** to ensure stable output.

In [4]:
DATASET_DIR = "dataset"
METADATA_FILE = os.path.join(DATASET_DIR, "metadata.jsonl")

# Use the proper special token for images
IMAGE_TOKEN = processor.image_token if hasattr(processor, 'image_token') else "<image>"

def convert_to_simple_format(sample):
    """
    Simple completion format: detect PII: <image>\nAnswer: <loc>...
    """
    full_text = f"detect PII: {IMAGE_TOKEN}\nAnswer: {sample['suffix']}"
    return {
        "text": full_text,
        "image_path": os.path.join(DATASET_DIR, "images", sample["image"])
    }

with open(METADATA_FILE, "r") as f:
    raw_dataset = [json.loads(line) for line in f]

formatted_dataset = []
for sample in raw_dataset:
    try:
        formatted_sample = convert_to_simple_format(sample)
        img = Image.open(formatted_sample["image_path"]).convert("RGB")
        formatted_dataset.append({
            "text": formatted_sample["text"],
            "image": img
        })
    except Exception as e:
        print(f"Error loading {sample['image']}: {e}")

print(f"Loaded {len(formatted_dataset)} samples.")

Loaded 206 samples.


## 5. Vision-aware Data Collator

In [5]:
def collate_fn(examples):
    texts = [ex["text"] for ex in examples]
    # Wrap each image in a list: Gemma 3 processor expects a list of images PER text prompt
    images = [[ex["image"]] for ex in examples]

    batch = processor(
        text=texts, 
        images=images, 
        return_tensors="pt", 
        padding=True
    ).to(model.device)

    labels = batch["input_ids"].clone()
    
    # Mask the prompt from labels
    for i, full_text in enumerate(texts):
        try:
            prompt_end_str = "\nAnswer: "
            prompt_part = full_text.split(prompt_end_str)[0] + prompt_end_str
            prompt_tokens = processor.tokenizer(prompt_part, add_special_tokens=False)["input_ids"]
            labels[i, :len(prompt_tokens)] = -100
        except:
            pass

    # Mask padding and special tokens
    labels[labels == processor.tokenizer.pad_token_id] = -100
    image_token_id = processor.tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
    labels[labels == image_token_id] = -100
    
    batch["labels"] = labels
    return batch

## 6. Training with SFTTrainer

In [6]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=32,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir="gemma3-pii-detector-v2",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=5,
    save_strategy="no",
    learning_rate=1e-4,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    dataset_text_field="", 
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to="none"
)

args.remove_unused_columns = False

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=formatted_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

print("Starting training... ")
trainer.train()

trainer.save_model("./final_gemma3_pii_model_v2")
print("Fine-tuning complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Starting training... 


ValueError: Prompt contained 0 image tokens but received 1 images.

## 7. Testing Inference

In [ ]:
def test_gemma3_vlm(image_path):
    image = Image.open(image_path).convert("RGB")
    prompt_text = f"detect PII: {IMAGE_TOKEN}\nAnswer:"
    
    # Inference also needs images to be a list of lists if text is a list of strings
    inputs = processor(text=[prompt_text], images=[[image]], return_tensors="pt").to(model.device)
    
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=128,
            do_sample=False,
            repetition_penalty=1.5,
            eos_token_id=processor.tokenizer.eos_token_id
        )
        
    input_len = inputs.input_ids.shape[1]
    result = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)
    
    print(f"Image: {image_path}")
    print(f"Prediction: {result}")
    return result

import glob
imgs = glob.glob("dataset/images/*.png")
if imgs:
    test_gemma3_vlm(random.choice(imgs))